# 07 · Scan serialized model artifacts **before** loading them

**Objective (15 min):** create two synthetic pickle files — one benign, one whose reducer would run a
command on unpickle — and show that a static scanner (ProtectAI **ModelScan**) finds the payload
without ever deserializing it. Then write the admission decision as code.

Pickling `SuspiciousObject` only serializes the *recipe*; nothing runs until someone calls
`pickle.load`. **We never do that here.**

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import json
import os
import pickle
import shlex
import subprocess
from hashlib import sha256
from pathlib import Path

import pandas as pd
from IPython.display import display

from workshop_utils import cli, require_package, save_json, sha256_file

require_package("modelscan")
MODELSCAN = cli("modelscan")     # resolved next to this kernel's interpreter, not from PATH

artifacts = Path("_evidence/model_artifacts")
artifacts.mkdir(parents=True, exist_ok=True)

In [ ]:
class SuspiciousObject:
    def __reduce__(self):
        return (os.system, ("echo MODEL_ARTIFACT_SHOULD_NEVER_EXECUTE",))

safe_path = artifacts / "safe_metadata.pkl"
suspicious_path = artifacts / "suspicious_model.pkl"

with safe_path.open("wb") as f:
    pickle.dump({"model_type": "demo", "weights": [0.1, 0.2]}, f)
with suspicious_path.open("wb") as f:
    pickle.dump(SuspiciousObject(), f)

print("Created files. They will NOT be unpickled in this notebook.")

## 1. Inventory: immutable digests first

Whatever the scanner says, the decision must be attached to a **digest**, not a filename.

In [ ]:
inventory = [{"path": str(p), "size_bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in [safe_path, suspicious_path]]
pd.DataFrame(inventory)

## 2. Run ModelScan in a subprocess

The scanner receives only the quarantined path. Do not import the artifact's Python package or
instantiate its model class first. ModelScan documents stable exit codes: **0** clean, **1**
vulnerabilities found, **2** scan error, **3** no supported files, **4** usage error.

In [ ]:
def scan(path: Path) -> dict:
    report = path.with_suffix(".modelscan.json")
    cmd = [MODELSCAN, "-p", str(path), "-r", "json", "-o", str(report)]
    print("$", shlex.join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
    parsed = json.loads(report.read_text(encoding="utf-8")) if report.exists() else None
    return {"path": str(path), "returncode": proc.returncode, "report": parsed,
            "stdout_tail": (proc.stdout + proc.stderr).strip()[-600:]}

scan_results = {p.name: scan(p) for p in [safe_path, suspicious_path]}
for name, r in scan_results.items():
    print(f"\n{name}: exit={r['returncode']}  issues={r['report']['summary']['total_issues'] if r['report'] else '?'}")
    for issue in (r["report"] or {}).get("issues", []):
        print("   ", issue["severity"], "-", issue["description"])

In [ ]:
assert scan_results["safe_metadata.pkl"]["returncode"] == 0
assert scan_results["suspicious_model.pkl"]["returncode"] == 1
critical = [i for i in scan_results["suspicious_model.pkl"]["report"]["issues"] if i["severity"] == "CRITICAL"]
assert critical and critical[0]["operator"] == "system", critical
print("PASS: the scanner found the os.system reducer without loading the artifact")

## 3. The Python API for an admission service

The CLI is easiest to gate in CI. The programmatic API is useful when an artifact-admission service
needs structured issue counts and severities.

In [ ]:
from modelscan.modelscan import ModelScan
from modelscan.settings import DEFAULT_SETTINGS

programmatic_results = []
for path in [safe_path, suspicious_path]:
    scanner = ModelScan(settings=DEFAULT_SETTINGS)
    scanner.scan(str(path))
    grouped = scanner.issues.group_by_severity()
    programmatic_results.append({
        "path": str(path),
        "issue_count": len(scanner.issues.all_issues),
        "issues_by_severity": {str(k): len(v) for k, v in grouped.items()},
        "errors": len(scanner.errors),
    })
display(pd.DataFrame(programmatic_results))
assert programmatic_results[0]["issue_count"] == 0
assert programmatic_results[1]["issue_count"] >= 1

## 4. Admission policy as code

Treat scanner errors, unsupported formats, and high-severity findings as **blocks**. A clean
serialization scan is one input; provenance, dependency vulnerabilities, custom code, resource limits,
license, and data lineage still require review. Prefer `safetensors` over pickle-based formats when
you control the producer.

In [ ]:
def admission_decision(scan: dict | None, provenance_verified: bool, format_supported: bool) -> dict:
    if not provenance_verified:
        return {"outcome": "block", "reason": "PROVENANCE_UNVERIFIED"}
    if not format_supported:
        return {"outcome": "block", "reason": "FORMAT_UNSUPPORTED"}
    if scan is None:
        return {"outcome": "block", "reason": "SCAN_NOT_RUN"}
    rc = scan["returncode"]
    if rc == 0:
        return {"outcome": "allow_to_staging", "reason": "STATIC_SCAN_CLEAR"}
    if rc == 1:
        return {"outcome": "block", "reason": "VULNERABILITY_FOUND"}
    if rc == 3:
        return {"outcome": "block", "reason": "NO_SUPPORTED_ARTIFACT"}
    return {"outcome": "block", "reason": "SCAN_ERROR"}

decisions = {
    "safe_metadata.pkl":    admission_decision(scan_results["safe_metadata.pkl"], provenance_verified=True, format_supported=True),
    "suspicious_model.pkl": admission_decision(scan_results["suspicious_model.pkl"], provenance_verified=True, format_supported=True),
    "unscanned":            admission_decision(None, provenance_verified=True, format_supported=True),
    "unknown_origin":       admission_decision(scan_results["safe_metadata.pkl"], provenance_verified=False, format_supported=True),
}
display(pd.DataFrame(decisions).T)
assert decisions["safe_metadata.pkl"]["outcome"] == "allow_to_staging"
assert all(d["outcome"] == "block" for k, d in decisions.items() if k != "safe_metadata.pkl")

In [ ]:
evidence = {
    "warning": "Files were created and scanned only; never deserialized.",
    "modelscan_version": scan_results["safe_metadata.pkl"]["report"]["summary"]["modelscan_version"],
    "inventory": inventory,
    "scan_results": {k: {kk: vv for kk, vv in v.items() if kk != "stdout_tail"} for k, v in scan_results.items()},
    "programmatic_results": programmatic_results,
    "admission_decisions": decisions,
}
out = save_json("_evidence/07_modelscan_evidence.json", evidence)
print("Wrote", out.resolve())

### Never do this

Do not add `pickle.load`, `torch.load`, `joblib.load`, or an equivalent loader to "confirm whether the
warning is real." Escalate the artifact into an isolated analysis environment or obtain a trusted,
verifiable replacement. The dependency lock file for this workshop (`requirements.txt` with
`--hash` lines, `uv.lock`) is the same idea applied to Python packages.